In [ ]:
%%capture
import os
import pandas as pd
from dj_notebook import activate
from pathlib import Path

env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)


In [ ]:
from intecomm_analytics.dataframes.main_1858_to_stata import to_stata, df_main_variable_labels
from intecomm_analytics.dataframes import get_df_main_1858
from intecomm_analytics.constants import HTN_DM, DM_ALONE, HTN_ALONE


In [ ]:
df_main = get_df_main_1858(None, fasting_hours=8.0)

In [ ]:
df_main[[col for col in df_main.columns if col.endswith("endline")]]


In [ ]:
[col for col in df_main.columns if col.endswith("endline")]

In [ ]:
# check variable_labels for 80 char limit
varlables = df_main_variable_labels()
# export to stata
to_stata(df_main, analysis_folder)

In [ ]:
# export a csv or counts for the primary variables, if needed
df = df_main.groupby(by=["primary_cohort_str"]).agg({col:"count" for col in df_main.columns if col.startswith("primary")})
df = pd.concat([df, pd.DataFrame([df.sum()])])
df = df.reset_index()

df_new = (
    df.melt(id_vars='index', var_name='variable', value_name='value')
    .pivot(index='variable', columns='index', values='value')
    .reset_index().reset_index(drop=True)
    .rename(columns={0:"TOTALS"})
)
df_new

In [ ]:
df_main

In [ ]:
df_main.query("primary_cohort_str=='HTN_DM'").groupby("assignment").size()

In [ ]:
from intecomm_analytics.constants import HTN_DM, DM_ALONE, HTN_ALONE

primary_cohort_mask = df_main["primary_cohort"].isin([HTN_ALONE, DM_ALONE, HTN_DM])
df = df_main[primary_cohort_mask].groupby(by=["primary_cohort_str", "assignment"]).size().to_frame().reset_index()
df.columns = ["primary_cohort_str", "assignment", "patients"]
df.pivot(index="primary_cohort_str", columns="assignment", values="patients")


In [ ]:
list(df_main.columns)

In [ ]:
df = df_main[primary_cohort_mask].groupby(by=["offstudy_reason", "assignment"]).size().to_frame().reset_index()
df.columns = ["offstudy_reason", "assignment", "patients"]
df = df.pivot(index="offstudy_reason", columns="assignment", values="patients").reset_index()
df[df["offstudy_reason"].isin(["LTFU", "consent_withdrawal", "dead", "clinical_withdrawal", "pregnant", "transferred"])].sum()

In [ ]:
df_main[primary_cohort_mask].groupby(by=["retained_12m", "assignment"]).size().to_frame().reset_index()

In [ ]:
htn_alone_mask = (df_main["primary_cohort"].isin([HTN_ALONE]) & (df_main["retained_12m"]==1))
controlled_mask = (df_main["bp_controlled_endline"]==1)
df_htn_alone = df_main[htn_alone_mask & controlled_mask].groupby(by=["assignment"]).size().to_frame().reset_index()
df_htn_alone["cat"] = "HTN_ALONE"
df_htn_alone

In [ ]:
df_main[htn_alone_mask].groupby(by=["assignment"]).size().to_frame().reset_index()


In [ ]:
dm_alone_mask = (df_main["primary_cohort"].isin([DM_ALONE]) & (df_main["retained_12m"]==1))
controlled_mask = (df_main["glucose_controlled_endline"]==1)
df_dm_alone = df_main[dm_alone_mask & controlled_mask].groupby(by=["assignment"]).size().to_frame().reset_index()
df_dm_alone["cat"] = "DM_ALONE"
df_dm_alone

In [ ]:
df_main[dm_alone_mask].groupby(by=["assignment"]).size().to_frame().reset_index()

In [ ]:
htn_dm_mask = (df_main["primary_cohort"].isin([HTN_DM]) & (df_main["retained_12m"]==1))
controlled_mask = ((df_main["glucose_controlled_endline"]==1) & (df_main["bp_controlled_endline"]==1))
df_htn_dm = df_main[htn_dm_mask & controlled_mask].groupby(by=["assignment"]).size().to_frame().reset_index().rename(columns={"glucose_controlled_endline": "controlled"})
df_htn_dm["cat"] = "HTN_DM"
df_htn_dm

In [ ]:
df_main[htn_dm_mask].groupby(by=["assignment"]).size().to_frame().reset_index()


In [ ]:
244+22+51

In [ ]:
241+28+35

In [ ]:
347+73+182

In [ ]:
htn_dm_mask = (df_main["primary_cohort"].isin([HTN_DM, DM_ALONE, HTN_ALONE]))
df_main[htn_dm_mask & (df_main["controlled_endline"]==1)].groupby(by=["assignment"]).size().to_frame().reset_index()


In [ ]:
df_main[htn_dm_mask & (df_main.referral==1)].groupby("assignment").size().to_frame().reset_index()